# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [2]:
%pip uninstall -y transformers peft huggingface_hub
%pip install --no-cache-dir -U \
    transformers \
    peft \
    huggingface_hub \
    datasets \
    accelerate \
    sentencepiece

Found existing installation: transformers 4.56.0
Uninstalling transformers-4.56.0:
  Successfully uninstalled transformers-4.56.0
Found existing installation: huggingface-hub 0.34.4
Uninstalling huggingface-hub-0.34.4:
  Successfully uninstalled huggingface-hub-0.34.4
Note: you may need to restart the kernel to use updated packages.
INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 181.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 197.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 167.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import gc
import math
import random
from pathlib import Path

import torch
import transformers
import datasets
import peft

from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

TRAIN_DATA_PATH = Path("/root/train.jsonl")
OUTPUT_DIR = Path("/root/inspect_pipeline_output")

MAX_LENGTH = 512

NUM_EXAMPLES = 8
NUM_EVAL_EXAMPLES = 2

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", repr(tokenizer.pad_token))
print("PAD token ID:", tokenizer.pad_token_id)
print("Padding side:", tokenizer.padding_side)
print("Chat template exists:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer class: Qwen2Tokenizer
Vocabulary size: 151643
EOS token: '<|im_end|>'
EOS token ID: 151645
PAD token: '<|endoftext|>'
PAD token ID: 151643
Padding side: right
Chat template exists: True


In [5]:
dataset = load_dataset(
    "json",
    data_files=str(TRAIN_DATA_PATH),
    split="train",
)

print(dataset)
print("Total conversations:", len(dataset))
print("Columns:", dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'source', 'num_turns'],
    num_rows: 1030
})
Total conversations: 1030
Columns: ['messages', 'source', 'num_turns']


Example keys: dict_keys(['messages', 'source', 'num_turns'])
Number of messages: 2

____________________________________________________________________________________________________
Message index: 0
Role: user
Content preview:
Can brain cells move? By movement I mean long distance migration (preferably within the brain only).

____________________________________________________________________________________________________
Message index: 1
Role: assistant
Content preview:
The question is relatively broad and one should take into account that the brain not only consists of neurons, but also glial cells (supportive cells) and pre-mitotic neuronal stem cells. Furthermore, as critical fellow-scientists have indicated, developmental stage is very important, as the developing embryonic brain is very different from the adult brain.
However, after sifting through various publications, the answer to the question is actually remarkably simple: Yes, brain cells migrate.
In 


In [7]:
invalid_examples = []

for example_index in range(len(dataset)):
    messages = dataset[example_index]["messages"]

    if len(messages) < 2:
        invalid_examples.append(
            {
                "index": example_index,
                "reason": "Conversation has fewer than two messages",
            }
        )
        continue

    if messages[-1]["role"] != "assistant":
        invalid_examples.append(
            {
                "index": example_index,
                "reason": "Final message is not from the assistant",
            }
        )

print("Invalid conversations:", len(invalid_examples))

for item in invalid_examples[:5]:
    print(item)

Invalid conversations: 1
{'index': 1021, 'reason': 'Final message is not from the assistant'}


In [8]:
if NUM_EXAMPLES > len(dataset):
    raise ValueError(
        f"Requested {NUM_EXAMPLES} examples, but the dataset "
        f"contains only {len(dataset)}."
    )

if NUM_EVAL_EXAMPLES >= NUM_EXAMPLES:
    raise ValueError(
        "NUM_EVAL_EXAMPLES must be smaller than NUM_EXAMPLES."
    )

small_dataset = (
    dataset
    .shuffle(seed=SEED)
    .select(range(NUM_EXAMPLES))
)

split_dataset = small_dataset.train_test_split(
    test_size=NUM_EVAL_EXAMPLES,
    seed=SEED,
)

raw_train_dataset = split_dataset["train"]
raw_eval_dataset = split_dataset["test"]

print("Selected conversations:", len(small_dataset))
print("Training conversations:", len(raw_train_dataset))
print("Evaluation conversations:", len(raw_eval_dataset))

Selected conversations: 8
Training conversations: 6
Evaluation conversations: 2


In [9]:
sample_messages = raw_train_dataset[0]["messages"]

formatted_conversation = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_conversation)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
$A$ and $B$ are $n \times n$ matrices and $v$ is a vector with $n$ elements. $Av$ has $\approx 2n^2$ flops and $A+B$ has $n^2$ flops. Following this logic, $(A+B)v$ should be faster than $Av+Bv$.
Yet, when I run the following code in matlab
```A = rand(2000,2000);
B = rand(2000,2000);
v = rand(2000,1);
tic
D=zeros(size(A));
D = A;
for i =1:100
    D = A + B;
    (D)*v;
end
toc
tic
for i =1:100
    (A*v+B*v);
end
toc
```
The opposite is true. Av+Bv is over twice as fast. Any explanations?<|im_end|>
<|im_start|>assistant
Except for code which does a significant number of floating-point operations on data that are held in cache, most floating-point intensive code is performance limited by memory bandwidth and cache capacity rather than by flops.
$v$ and the products $Av$ and $Bv$ are all vectors of length 2000 (16K bytes in double precision), which will easily fit into a leve

In [10]:
prompt_messages = sample_messages[:-1]

formatted_prompt = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(formatted_prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
$A$ and $B$ are $n \times n$ matrices and $v$ is a vector with $n$ elements. $Av$ has $\approx 2n^2$ flops and $A+B$ has $n^2$ flops. Following this logic, $(A+B)v$ should be faster than $Av+Bv$.
Yet, when I run the following code in matlab
```A = rand(2000,2000);
B = rand(2000,2000);
v = rand(2000,1);
tic
D=zeros(size(A));
D = A;
for i =1:100
    D = A + B;
    (D)*v;
end
toc
tic
for i =1:100
    (A*v+B*v);
end
toc
```
The opposite is true. Av+Bv is over twice as fast. Any explanations?<|im_end|>
<|im_start|>assistant



In [12]:
print("_" * 100)
print("PROMPT PORTION")
print("_" * 100)
print(formatted_prompt)

print()
print("_" * 100)
print("COMPLETE TRAINING CONVERSATION")
print("_" * 100)
print(formatted_conversation)

____________________________________________________________________________________________________
PROMPT PORTION
____________________________________________________________________________________________________
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
$A$ and $B$ are $n \times n$ matrices and $v$ is a vector with $n$ elements. $Av$ has $\approx 2n^2$ flops and $A+B$ has $n^2$ flops. Following this logic, $(A+B)v$ should be faster than $Av+Bv$.
Yet, when I run the following code in matlab
```A = rand(2000,2000);
B = rand(2000,2000);
v = rand(2000,1);
tic
D=zeros(size(A));
D = A;
for i =1:100
    D = A + B;
    (D)*v;
end
toc
tic
for i =1:100
    (A*v+B*v);
end
toc
```
The opposite is true. Av+Bv is over twice as fast. Any explanations?<|im_end|>
<|im_start|>assistant


____________________________________________________________________________________________________
COMPLETE TRAINING CONVERSATION
_________

In [13]:
def tokenize_conversation(example):
    messages = example["messages"]

    if len(messages) < 2:
        raise ValueError(
            "Each conversation must contain at least two messages."
        )

    if messages[-1]["role"] != "assistant":
        raise ValueError(
            "The final message must come from the assistant."
        )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )

    full_input_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]

    prompt_input_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    prompt_length = len(prompt_input_ids)

    if full_input_ids[:prompt_length] != prompt_input_ids:
        raise ValueError(
            "The prompt token IDs do not match the beginning "
            "of the complete conversation."
        )

    labels = (
        [-100] * prompt_length
        + full_input_ids[prompt_length:]
    )

    original_length = len(full_input_ids)

    if original_length > MAX_LENGTH:
        full_input_ids = full_input_ids[-MAX_LENGTH:]
        labels = labels[-MAX_LENGTH:]

    attention_mask = [1] * len(full_input_ids)

    if not (
        len(full_input_ids)
        == len(attention_mask)
        == len(labels)
    ):
        raise ValueError(
            "input_ids, attention_mask and labels have "
            "different lengths."
        )

    if all(label == -100 for label in labels):
        raise ValueError(
            "No supervised assistant tokens remain after truncation."
        )

    return {
        "input_ids": full_input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "original_length": original_length,
    }

In [14]:
tokenized_sample = tokenize_conversation(
    raw_train_dataset[0]
)

print("Original length:", tokenized_sample["original_length"])
print("Final input length:", len(tokenized_sample["input_ids"]))
print("Attention-mask length:", len(tokenized_sample["attention_mask"]))
print("Labels length:", len(tokenized_sample["labels"]))

Original length: 674
Final input length: 512
Attention-mask length: 512
Labels length: 512


In [15]:
ignored_token_count = 0
supervised_token_count = 0

for label in tokenized_sample["labels"]:
    if label == -100:
        ignored_token_count += 1
    else:
        supervised_token_count += 1

print("Total tokens:", len(tokenized_sample["labels"]))
print("Ignored tokens:", ignored_token_count)
print("Supervised tokens:", supervised_token_count)

Total tokens: 512
Ignored tokens: 71
Supervised tokens: 441


In [16]:
complete_text = tokenizer.decode(
    tokenized_sample["input_ids"],
    skip_special_tokens=False,
)

print(complete_text)

 i =1:100
    D = A + B;
    (D)*v;
end
toc
tic
for i =1:100
    (A*v+B*v);
end
toc
```
The opposite is true. Av+Bv is over twice as fast. Any explanations?<|im_end|>
<|im_start|>assistant
Except for code which does a significant number of floating-point operations on data that are held in cache, most floating-point intensive code is performance limited by memory bandwidth and cache capacity rather than by flops.
$v$ and the products $Av$ and $Bv$ are all vectors of length 2000 (16K bytes in double precision), which will easily fit into a level 1 cache.  The matrices $A$ and $B$ are 2000 by 2000 or about 32 megabytes in size.  Your level 3 cache might be large enough to store one of these matrices if you've got a really good processor.

Computing $Av$ requires reading 32 megabytes (for $A$) in from memory, reading in 16K bytes (for $v$) storing intermediate results in the L1 cache and eventually writing 16K bytes out to memory. Multiplying $Bv$ takes the same amount of work. Adding the

In [17]:
supervised_token_ids = []

for input_id, label in zip(
    tokenized_sample["input_ids"],
    tokenized_sample["labels"],
):
    if label != -100:
        supervised_token_ids.append(input_id)

supervised_text = tokenizer.decode(
    supervised_token_ids,
    skip_special_tokens=False,
)

print(supervised_text)

Except for code which does a significant number of floating-point operations on data that are held in cache, most floating-point intensive code is performance limited by memory bandwidth and cache capacity rather than by flops.
$v$ and the products $Av$ and $Bv$ are all vectors of length 2000 (16K bytes in double precision), which will easily fit into a level 1 cache.  The matrices $A$ and $B$ are 2000 by 2000 or about 32 megabytes in size.  Your level 3 cache might be large enough to store one of these matrices if you've got a really good processor.

Computing $Av$ requires reading 32 megabytes (for $A$) in from memory, reading in 16K bytes (for $v$) storing intermediate results in the L1 cache and eventually writing 16K bytes out to memory. Multiplying $Bv$ takes the same amount of work. Adding the two intermediate results to get the final result requires a trivial amount of work.  That's a total of roughly 64 megabytes of reads and an insignificant number of writes.

Computing $(A+B

In [18]:
first_supervised_index = None

for i in range(len(tokenized_sample["labels"])):
    if tokenized_sample["labels"][i] != -100:
        first_supervised_index = i
        break

print("First supervised index:", first_supervised_index)

start_index = max(0, first_supervised_index - 10)
end_index = min(
    len(tokenized_sample["input_ids"]),
    first_supervised_index + 20,
)

print()
print(
    f"{'Index':<8}"
    f"{'Input ID':<12}"
    f"{'Label':<12}"
    f"Token"
)
print("-" * 80)

for i in range(start_index, end_index):
    input_id = tokenized_sample["input_ids"][i]
    label = tokenized_sample["labels"][i]
    token = tokenizer.decode([input_id])

    print(
        f"{i:<8}"
        f"{input_id:<12}"
        f"{label:<12}"
        f"{repr(token)}"
    )

First supervised index: 71

Index   Input ID    Label       Token
--------------------------------------------------------------------------------
61      4937        -100        ' fast'
62      13          -100        '.'
63      5765        -100        ' Any'
64      40841       -100        ' explanations'
65      30          -100        '?'
66      151645      -100        '<|im_end|>'
67      198         -100        '\n'
68      151644      -100        '<|im_start|>'
69      77091       -100        'assistant'
70      198         -100        '\n'
71      58015       58015       'Except'
72      369         369         ' for'
73      2038        2038        ' code'
74      892         892         ' which'
75      1558        1558        ' does'
76      264         264         ' a'
77      5089        5089        ' significant'
78      1372        1372        ' number'
79      315         315         ' of'
80      19057       19057       ' floating'
81      16574       16574       '-p

In [19]:
train_dataset = raw_train_dataset.map(
    tokenize_conversation,
    remove_columns=raw_train_dataset.column_names,
    desc="Tokenizing training conversations",
)

eval_dataset = raw_eval_dataset.map(
    tokenize_conversation,
    remove_columns=raw_eval_dataset.column_names,
    desc="Tokenizing evaluation conversations",
)

print(train_dataset)
print(eval_dataset)

Tokenizing training conversations:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing evaluation conversations:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'original_length'],
    num_rows: 6
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'original_length'],
    num_rows: 2
})


In [20]:
train_original_lengths = train_dataset["original_length"]
eval_original_lengths = eval_dataset["original_length"]

train_final_lengths = [
    len(example["input_ids"])
    for example in train_dataset
]

eval_final_lengths = [
    len(example["input_ids"])
    for example in eval_dataset
]

print("Training original lengths:", train_original_lengths)
print("Training final lengths:", train_final_lengths)

print()

print("Evaluation original lengths:", eval_original_lengths)
print("Evaluation final lengths:", eval_final_lengths)

Training original lengths: Column([674, 954, 1026, 168, 1227, ...])
Training final lengths: [512, 512, 512, 168, 512, 403]

Evaluation original lengths: Column([446, 1135])
Evaluation final lengths: [446, 512]


In [21]:
train_dataset = train_dataset.remove_columns(
    ["original_length"]
)

eval_dataset = eval_dataset.remove_columns(
    ["original_length"]
)

print("Training columns:", train_dataset.column_names)
print("Evaluation columns:", eval_dataset.column_names)

Training columns: ['input_ids', 'attention_mask', 'labels']
Evaluation columns: ['input_ids', 'attention_mask', 'labels']


In [22]:
class AssistantOnlyDataCollator:
    def __init__(
        self,
        tokenizer,
        pad_to_multiple_of=None,
    ):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        model_features = []

        for feature in features:
            model_features.append(
                {
                    "input_ids": feature["input_ids"],
                    "attention_mask": feature["attention_mask"],
                }
            )

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        padded_sequence_length = batch["input_ids"].shape[1]
        padded_labels = []

        for feature in features:
            labels = feature["labels"]

            padding_length = (
                padded_sequence_length - len(labels)
            )

            padded_example_labels = (
                labels
                + [-100] * padding_length
            )

            padded_labels.append(
                padded_example_labels
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch

In [23]:
data_collator = AssistantOnlyDataCollator(
    tokenizer=tokenizer,
)

batch = data_collator(
    [
        train_dataset[0],
        train_dataset[1],
    ]
)

print("Batch keys:", batch.keys())
print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

Batch keys: KeysView({'input_ids': tensor([[   600,    284,     16,  ...,    981, 151645,    198],
        [   805,   1033,   4558,  ...,     13, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([[  -100,   -100,   -100,  ...,    981, 151645,    198],
        [   805,   1033,   4558,  ...,     13, 151645,    198]])})
Input IDs shape: torch.Size([2, 512])
Attention mask shape: torch.Size([2, 512])
Labels shape: torch.Size([2, 512])


In [24]:
for i in range(batch["input_ids"].shape[0]):
    total_length = batch["input_ids"].shape[1]

    real_token_count = int(
        batch["attention_mask"][i].sum().item()
    )

    padding_token_count = (
        total_length - real_token_count
    )

    supervised_token_count = int(
        (batch["labels"][i] != -100)
        .sum()
        .item()
    )

    print(f"Example {i}")
    print("  Total padded length:", total_length)
    print("  Real tokens:", real_token_count)
    print("  Padding tokens:", padding_token_count)
    print("  Supervised tokens:", supervised_token_count)
    print()

Example 0
  Total padded length: 512
  Real tokens: 512
  Padding tokens: 0
  Supervised tokens: 441

Example 1
  Total padded length: 512
  Real tokens: 512
  Padding tokens: 0
  Supervised tokens: 512



In [25]:
for i in range(batch["input_ids"].shape[0]):
    padding_positions = (
        batch["attention_mask"][i] == 0
    )

    padding_labels = batch["labels"][i][
        padding_positions
    ]

    if padding_labels.numel() > 0:
        assert torch.all(
            padding_labels == -100
        ), "A padding token has a non -100 label."

In [26]:
del batch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODEL_DTYPE = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.float32
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.float32

print("Temporary batch removed.")
print("Selected device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Temporary batch removed.
Selected device: cuda
Model dtype: torch.bfloat16
GPU: Tesla T4


In [27]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
)

model = model.to(DEVICE)

print("Model class:", type(model).__name__)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("Total parameters:", f"{model.num_parameters():,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model class: Qwen2ForCausalLM
Model device: cuda:0
Model dtype: torch.bfloat16
Total parameters: 494,032,768


In [28]:
TEST_PROMPT = (
    "Explain supervised fine-tuning in simple words "
    "and give one example."
)


def generate_response(
    current_model,
    prompt,
    max_new_tokens=100,
):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    model_inputs = {
        key: value.to(DEVICE)
        for key, value in model_inputs.items()
    }

    current_model.eval()
    current_model.config.use_cache = True

    with torch.no_grad():
        generated_ids = current_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = model_inputs["input_ids"].shape[1]

    response_ids = generated_ids[
        0,
        prompt_length:,
    ]

    response = tokenizer.decode(
        response_ids,
        skip_special_tokens=True,
    )

    return response.strip()

In [29]:
base_response = generate_response(
    current_model=model,
    prompt=TEST_PROMPT,
)

print("_" * 100)
print("PROMPT")
print("_" * 100)
print(TEST_PROMPT)

print()
print("_" * 100)
print("BASE MODEL RESPONSE")
print("_" * 100)
print(base_response)

____________________________________________________________________________________________________
PROMPT
____________________________________________________________________________________________________
Explain supervised fine-tuning in simple words and give one example.

____________________________________________________________________________________________________
BASE MODEL RESPONSE
____________________________________________________________________________________________________
Supervised fine-tuning is a technique used in machine learning where the model is trained on a large dataset of labeled data to learn how to perform specific tasks. The goal is to make predictions or decisions based on new, unseen data.

In this process, we first train our model using a large amount of labeled data that includes both examples with known labels (training set) and examples without labels (testing set). The model learns from these training examples to recognize patterns and relati

In [30]:
base_evaluation_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "base_evaluation"),

    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    disable_tqdm=True,

    use_cpu=(DEVICE.type == "cpu"),

    seed=SEED,
)

In [31]:
base_trainer = Trainer(
    model=model,
    args=base_evaluation_args,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

base_metrics = base_trainer.evaluate()

base_eval_loss = base_metrics["eval_loss"]
base_perplexity = math.exp(base_eval_loss)

print("Base evaluation loss:", base_eval_loss)
print("Base perplexity:", base_perplexity)

[RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'eval_loss': '2.29', 'eval_model_preparation_time': '0.0048', 'eval_runtime': '0.5511', 'eval_samples_per_second': '3.629', 'eval_steps_per_second': '3.629', 'epoch': 0}
Base evaluation loss: 2.2899973392486572
Base perplexity: 9.874911406454444


Base evaluation Trainer removed.


LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'o_proj', 'v_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Trainable parameters: 1,081,344
Total parameters: 495,114,112
Trainable percentage: 0.2184%


Training examples: 6
Per-device batch size: 1
Gradient accumulation steps: 2
Effective batch size: 2
Estimated optimiser steps: 3


In [38]:
model.config.use_cache = False

print("use_cache:", model.config.use_cache)

use_cache: False


In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer created.")

[RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer created.


In [40]:
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


{'loss': '2.353', 'grad_norm': '1.064', 'learning_rate': '0.0002', 'epoch': '0.3333'}
{'loss': '2.644', 'grad_norm': '0.7435', 'learning_rate': '0.0001333', 'epoch': '0.6667'}
{'loss': '2.177', 'grad_norm': '0.9266', 'learning_rate': '6.667e-05', 'epoch': '1'}
{'train_runtime': '3.1', 'train_samples_per_second': '1.936', 'train_steps_per_second': '0.968', 'train_loss': '2.391', 'epoch': '1'}


In [41]:
print("Training metrics")

for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

Training metrics
train_runtime: 3.0998
train_samples_per_second: 1.936
train_steps_per_second: 0.968
train_loss: 2.3911940256754556
epoch: 1.0


In [42]:
training_loss_history = []

for log in trainer.state.log_history:
    if "loss" in log:
        training_loss_history.append(
            {
                "step": log.get("step"),
                "epoch": log.get("epoch"),
                "loss": log.get("loss"),
                "learning_rate": log.get("learning_rate"),
                "grad_norm": log.get("grad_norm"),
            }
        )

print(
    f"{'Step':<8}"
    f"{'Epoch':<12}"
    f"{'Loss':<14}"
    f"{'Learning rate':<18}"
    f"{'Gradient norm':<16}"
)

print("_" * 100)

for record in training_loss_history:
    print(
        f"{record['step']:<8}"
        f"{record['epoch']:<12.4f}"
        f"{record['loss']:<14.4f}"
        f"{record['learning_rate']:<18.8f}"
        f"{str(record['grad_norm']):<16}"
    )

Step    Epoch       Loss          Learning rate     Gradient norm   
____________________________________________________________________________________________________
1       0.3333      2.3531        0.00020000        1.064108967781067
2       0.6667      2.6439        0.00013333        0.74349045753479
3       1.0000      2.1766        0.00006667        0.9265608787536621


In [43]:
sft_metrics = trainer.evaluate()

sft_eval_loss = sft_metrics["eval_loss"]
sft_perplexity = math.exp(sft_eval_loss)

print("SFT evaluation loss:", sft_eval_loss)
print("SFT perplexity:", sft_perplexity)

{'eval_loss': '2.284', 'eval_runtime': '0.5427', 'eval_samples_per_second': '3.685', 'eval_steps_per_second': '3.685', 'epoch': '1'}
SFT evaluation loss: 2.284189462661743
SFT perplexity: 9.817725365181309


In [45]:
loss_difference = (
    sft_eval_loss - base_eval_loss
)

perplexity_difference = (
    sft_perplexity - base_perplexity
)

print(
    f"{'Model':<20}"
    f"{'Evaluation loss':<20}"
    f"{'Perplexity':<20}"
)

print("_" * 100)

print(
    f"{'Base Qwen':<20}"
    f"{base_eval_loss:<20.4f}"
    f"{base_perplexity:<20.4f}"
)

print(
    f"{'LIMA SFT Qwen':<20}"
    f"{sft_eval_loss:<20.4f}"
    f"{sft_perplexity:<20.4f}"
)

print()
print("Loss difference:", round(loss_difference, 4))
print(
    "Perplexity difference:",
    round(perplexity_difference, 4),
)

Model               Evaluation loss     Perplexity          
____________________________________________________________________________________________________
Base Qwen           2.2900              9.8749              
LIMA SFT Qwen       2.2842              9.8177              

Loss difference: -0.0058
Perplexity difference: -0.0572


In [47]:
model.config.use_cache = True

sft_response = generate_response(
    current_model=model,
    prompt=TEST_PROMPT,
)

print("_" * 100)
print("SFT MODEL RESPONSE")
print("_" * 100)
print(sft_response)

____________________________________________________________________________________________________
SFT MODEL RESPONSE
____________________________________________________________________________________________________
Supervised fine-tuning is a technique used in machine learning where we train a model on a large dataset of labeled data to improve its performance on new, unseen data. The goal is to make the model better at predicting or classifying outcomes based on input features.

In this process, we first use a pre-trained model (often a pre-trained language model like BERT) that has been trained on a large corpus of text data. This model can be very effective for tasks such as translation, summar


In [48]:
print("_" * 100)
print("PROMPT")
print("_" * 100)
print(TEST_PROMPT)

print()
print("_" * 100)
print("BASE MODEL RESPONSE")
print("_" * 100)
print(base_response)

print()
print("_" * 100)
print("LIMA SFT MODEL RESPONSE")
print("_" * 100)
print(sft_response)

____________________________________________________________________________________________________
PROMPT
____________________________________________________________________________________________________
Explain supervised fine-tuning in simple words and give one example.

____________________________________________________________________________________________________
BASE MODEL RESPONSE
____________________________________________________________________________________________________
Supervised fine-tuning is a technique used in machine learning where the model is trained on a large dataset of labeled data to learn how to perform specific tasks. The goal is to make predictions or decisions based on new, unseen data.

In this process, we first train our model using a large amount of labeled data that includes both examples with known labels (training set) and examples without labels (testing set). The model learns from these training examples to recognize patterns and relati

In [49]:
ADAPTER_OUTPUT_DIR = (
    OUTPUT_DIR
    / "inspection_adapter"
)

model.save_pretrained(
    ADAPTER_OUTPUT_DIR
)

tokenizer.save_pretrained(
    ADAPTER_OUTPUT_DIR
)

print(
    "Adapter saved to:",
    ADAPTER_OUTPUT_DIR.resolve(),
)

Adapter saved to: /root/inspect_pipeline_output/inspection_adapter


In [50]:
print("SFT pipeline inspection completed.")
print()

print("Device:", DEVICE)
print("Training conversations:", len(train_dataset))
print("Evaluation conversations:", len(eval_dataset))
print("Maximum sequence length:", MAX_LENGTH)

print()

print(
    "Trainable parameters:",
    f"{trainable_parameter_count:,}",
)

print(
    "Trainable percentage:",
    f"{trainable_percentage:.4f}%",
)

print()

print(
    "Base evaluation loss:",
    round(base_eval_loss, 4),
)

print(
    "SFT evaluation loss:",
    round(sft_eval_loss, 4),
)

print(
    "Base perplexity:",
    round(base_perplexity, 4),
)

print(
    "SFT perplexity:",
    round(sft_perplexity, 4),
)

print()

print(
    "Saved adapter:",
    ADAPTER_OUTPUT_DIR.resolve(),
)

SFT pipeline inspection completed.

Device: cuda
Training conversations: 6
Evaluation conversations: 2
Maximum sequence length: 512

Trainable parameters: 1,081,344
Trainable percentage: 0.2184%

Base evaluation loss: 2.29
SFT evaluation loss: 2.2842
Base perplexity: 9.8749
SFT perplexity: 9.8177

Saved adapter: /root/inspect_pipeline_output/inspection_adapter
